# 09 · Does a multi-season composite resolve the cropland/grassland confusion?

The first entry's clustering result (ARI = 0.275 vs. ESA WorldCover) was clean for Built-up and
Tree cover but diffuse for Cropland and Grassland, which spread across several clusters instead
of concentrating in one or two. The leading explanation, backed by the spectral-index literature
(Section 2.1 of the paper): cropland and grassland look nearly identical in a single mid-summer
snapshot and differ mainly in *phenology* -- crop calendar, irrigation timing, harvest -- which
one date can't capture. That was always stated as a plausible explanation, never actually tested.

This notebook tests it directly. Notebook 06 already fetched a winter mosaic over this exact
725-chip AOI and grid to measure seasonal *shift*, but never exported the raw winter embedding
vectors (only a scalar shift per chip) -- so this notebook redoes that fetch-and-embed step, this
time keeping the actual embeddings, and builds a **composite embedding** per chip by
concatenating its summer vector (already committed in `docs/data/embeddings.bin`) with its
winter vector. If phenology is really the missing signal, KMeans clustering on the 2048-dim
composite should separate Cropland and Grassland more cleanly than the original 1024-dim
summer-only clustering did -- measured against the exact same already-committed per-chip cluster
labels, not a re-run, for a controlled before/after comparison.

**Requires a GPU runtime.** Unlike notebooks 05/08's small ~50-chip AOIs, this re-embeds the full
725-chip grid for a second season, so it's a much bigger job -- plan for a real T4 run, not a CPU
fallback (technically possible via the same `torch.cuda.is_available()` pattern as notebook 08,
but 725 chips on CPU would likely take a long time). No Drive mount needed either way -- this
reads the summer baseline straight from committed `docs/data/` files.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import json

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import adjusted_rand_score

from src import clay_embed, stac_utils, viz_utils

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print("No GPU detected -- continuing on CPU. This re-embeds the full 725-chip grid for a")
    print("second season, which will be considerably slower on CPU than notebook 08's small AOI.")

os.makedirs("docs/figures", exist_ok=True)

with open("docs/data/chips.geojson") as f:
    chips_geojson = json.load(f)
features = chips_geojson["features"]

with open("docs/data/embeddings.bin", "rb") as f:
    flat = np.frombuffer(f.read(), dtype="<f4")
emb_summer = flat.reshape(len(features), -1)

print(f"Loaded {len(features)} chips, {emb_summer.shape[1]}-dim summer baseline embeddings")

## Fetch a winter mosaic over the same AOI

Identical to notebook 06's fetch: same bbox/CRS/resolution as notebook 01 so the pixel grid and
chip ids line up exactly, same per-tile least-cloudy selection for the two-MGRS-tile AOI, snow
left in on purpose since it's part of the real winter state.

In [ ]:
catalog = stac_utils.open_catalog()
WINTER_RANGE = "2023-12-01/2024-02-28"

winter_items_all = stac_utils.search_sentinel2(catalog, datetime_range=WINTER_RANGE, max_cloud_cover=30.0)
print(f"{len(winter_items_all)} candidate winter scenes")
winter_items = stac_utils.select_least_cloudy_per_tile(winter_items_all)

winter_ds = odc.stac.load(
    winter_items, bands=stac_utils.S2_BANDS, bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613", resolution=stac_utils.GSD_M, groupby="solar_day",
    chunks={"x": 1024, "y": 1024},
)
winter_mosaic = winter_ds.to_array(dim="band").median(dim="time").compute()
print(winter_mosaic.shape, winter_mosaic.dtype)

## Match winter pixels to the existing chip ids

In [ ]:
def centroid_of(feature):
    coords = feature["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return (min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2

height, width = winter_mosaic.sizes["y"], winter_mosaic.sizes["x"]
grid_by_id = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(height, width)}

NODATA_FRAC_THRESHOLD = 0.05
matched_idx, winter_pixels, lats, lons = [], [], [], []
for i, f in enumerate(features):
    win = grid_by_id.get(f["properties"]["id"])
    if win is None:
        continue
    patch = winter_mosaic.values[:, win["y_slice"], win["x_slice"]]
    if np.isnan(patch).mean() > NODATA_FRAC_THRESHOLD:
        continue
    lat, lon = centroid_of(f)
    matched_idx.append(i)
    winter_pixels.append(np.nan_to_num(patch, nan=0.0).astype("float32"))
    lats.append(lat)
    lons.append(lon)

winter_pixels = np.stack(winter_pixels)
matched_idx = np.array(matched_idx)
print(f"Matched {len(matched_idx)}/{len(features)} chips between the summer baseline and this winter mosaic")

## Embed winter chips with Clay

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
metadata_path = clay_embed.download_metadata_yaml()
model = clay_embed.load_model(ckpt_path, metadata_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats(metadata_path)

WINTER_DATE = pd.Timestamp("2024-01-15")  # representative date within WINTER_RANGE, matches notebook 06

def embed_batch(pixels, date, lats, lons, batch_size=16):
    dates = [date] * len(lats)
    out = []
    for start in range(0, len(lats), batch_size):
        end = min(start + batch_size, len(lats))
        batch_pixels = clay_embed.normalize_chips(pixels[start:end], band_means, band_stds)
        time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(
            dates[start:end], lats[start:end], lons[start:end]
        )
        batch_emb = clay_embed.encode_batch(model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device)
        out.append(batch_emb)
        del batch_pixels, time_feats, latlon_feats
        if device == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(out, axis=0)

emb_winter = embed_batch(winter_pixels, WINTER_DATE, lats, lons)
print(f"Embedded {len(matched_idx)} winter chips, dim={emb_winter.shape[1]}")

## Build the composite embedding, and recluster

Concatenate each matched chip's summer and winter vectors into one 2048-dim composite. This is
a deliberately simple construction -- no learned fusion, just "give KMeans both dates' raw
information at once" -- so any improvement can be attributed to the phenology information itself
being present, not to a more sophisticated combination method.

In [ ]:
emb_summer_matched = emb_summer[matched_idx]
emb_composite = np.concatenate([emb_summer_matched, emb_winter], axis=1)
print(f"Composite embedding: {emb_composite.shape}")

composite_clusters = viz_utils.cluster_embeddings(emb_composite, method="kmeans", n_clusters=8)
print(f"Composite cluster sizes: {np.bincount(composite_clusters[composite_clusters >= 0])}")

## Match both clusterings to the same WorldCover ground truth

The "before" clustering here isn't a fresh re-run -- it's the exact cluster label already
committed per chip in `docs/data/chips.geojson` from the original summer-only analysis, restricted
to the same matched chip subset used for the composite. That keeps the comparison controlled: any
difference reflects the extra winter information, not a different random seed or a different set
of chips.

In [ ]:
WORLDCOVER_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
    90: "Wetland", 95: "Mangroves", 100: "Moss/lichen",
}

wc_items = stac_utils.search_worldcover(catalog)
wc_ds = odc.stac.load(
    wc_items, bbox=stac_utils.FRONT_RANGE_BBOX, crs="EPSG:32613",
    resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
)
wc_var = wc_ds["map"]
wc_map = wc_var.isel(time=0).compute().values if "time" in wc_ds.dims else wc_var.compute().values
wc_grid = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(*wc_map.shape)}

matched_features = [features[i] for i in matched_idx]
original_clusters = np.array([f["properties"]["cluster"] for f in matched_features])

majority_class, keep_mask = [], []
for f in matched_features:
    win = wc_grid.get(f["properties"]["id"])
    if win is None:
        keep_mask.append(False)
        continue
    patch = wc_map[win["y_slice"], win["x_slice"]]
    values, counts = np.unique(patch[patch > 0], return_counts=True)
    if len(values) == 0:
        keep_mask.append(False)
        continue
    majority_class.append(values[np.argmax(counts)])
    keep_mask.append(True)

keep_mask = np.array(keep_mask)
majority_class = np.array(majority_class)
class_names = np.array([WORLDCOVER_CLASSES.get(c, str(c)) for c in majority_class])

original_clusters_matched = original_clusters[keep_mask]
composite_clusters_matched = composite_clusters[keep_mask]

print(f"Matched {keep_mask.sum()}/{len(matched_idx)} chips to a WorldCover majority class")

ari_original = adjusted_rand_score(majority_class, original_clusters_matched)
ari_composite = adjusted_rand_score(majority_class, composite_clusters_matched)
print(f"\nARI, single-season summer-only clustering (this matched subset): {ari_original:.3f}")
print(f"ARI, two-season composite clustering:                              {ari_composite:.3f}")
print("(Original entry's full-725-chip ARI was 0.275 -- expect a close but not identical number")
print(" here since this subset excludes any chip that didn't survive the winter nodata filter.)")

## Did Cropland and Grassland specifically get cleaner?

The headline ARI is a blunt instrument -- the real question is narrower: do Cropland and
Grassland chips, which previously spread across several clusters, now concentrate into fewer of
them? For each class, "purity" here means the fraction of that class's chips landing in its own
single most common cluster -- higher means cleaner separation.

In [ ]:
def modal_cluster_purity(cluster_labels, class_mask):
    labels = cluster_labels[class_mask]
    if len(labels) == 0:
        return np.nan, 0
    values, counts = np.unique(labels, return_counts=True)
    return counts.max() / len(labels), len(labels)

for cls in ["Cropland", "Grassland"]:
    mask = class_names == cls
    p_before, n = modal_cluster_purity(original_clusters_matched, mask)
    p_after, _ = modal_cluster_purity(composite_clusters_matched, mask)
    print(f"{cls} (n={n}):")
    print(f"  single-season purity:  {p_before:.3f}")
    print(f"  composite purity:      {p_after:.3f}")
    print(f"  change:                {p_after - p_before:+.3f}")
    print()

print("A positive change for both classes supports the phenology explanation: adding a second")
print("season's information lets clustering separate them more cleanly. A flat or negative change")
print("means single-date phenology blindness wasn't the (whole) story.")

In [ ]:
contingency_composite = pd.crosstab(
    pd.Series(composite_clusters_matched, name="composite embedding cluster"),
    pd.Series(class_names, name="WorldCover class"),
)

plt.figure(figsize=(10, 6))
plt.imshow(contingency_composite.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(contingency_composite.columns)), contingency_composite.columns, rotation=45, ha="right")
plt.yticks(range(len(contingency_composite.index)), contingency_composite.index)
plt.xlabel("ESA WorldCover class")
plt.ylabel("Composite embedding cluster")
plt.title("Two-season composite clusters vs. ground-truth land cover")
plt.colorbar(label="# chips")
plt.tight_layout()
plt.savefig("docs/figures/composite_cluster_vs_worldcover.png", dpi=150)
plt.show()
contingency_composite

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
labels = ["Cropland", "Grassland"]
before_vals = [modal_cluster_purity(original_clusters_matched, class_names == c)[0] for c in labels]
after_vals = [modal_cluster_purity(composite_clusters_matched, class_names == c)[0] for c in labels]

x = np.arange(len(labels))
width = 0.35
ax.bar(x - width/2, before_vals, width, label="single-season (summer only)", color="#f58231")
ax.bar(x + width/2, after_vals, width, label="two-season composite", color="#5ec8ff")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel("modal-cluster purity")
ax.set_title("Does a second season sharpen cropland/grassland separation?")
ax.legend()
plt.tight_layout()
plt.savefig("docs/figures/composite_cropland_grassland_purity.png", dpi=150)
plt.show()

print("\nDone. Commit the new docs/figures/composite_*.png files back to the repo.")
print("(This notebook doesn't touch docs/data/chips.geojson -- composite clusters are analysis-only,")
print(" not exported to the live map's default single-season coloring.)")